TUTORIAL - Aprendizaje Profundo con YOLO
==========================================================

Este notebook tiene como objetivo familiarizarnos con el flujo de trabajo del proyecto. Se realizarán las siguientes tareas sobre la imagen **ID 139** del conjunto de validación COCO 2017:
1. Visualizar la máscara de verdad (Ground Truth) utilizando las anotaciones originales.
2. Ejecutar inferencias con cinco variantes de YOLOv8 (n, s, m, l, x).
3. Comparar rendimiento entre modelos (F1-Score).
4. Visualizar gráficamente las detecciones frente a la realidad.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import os
from ultralytics import YOLO
import numpy as np
import pandas as pd

### Funciones auxiliares para calcular F1-Score
Calcular el mAP (mean Average Precision) para una sola imagen es técnicamente posible pero poco significativo (es una estadística diseñada para conjuntos grandes). Sin embargo, el F1-Score es perfecto para esto, ya que mide el equilibrio entre precisión y exhaustividad (recall) en esa imagen concreta.

Por ello, implementamos una función para calcular el F1-Score comparando cada caja predicha con las del Ground Truth mediante IoU (Intersección sobre Unión).

In [ ]:
def calcular_iou(box1, box2):
    """Calcula Intersection over Union (IoU) entre dos cajas [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0


def calcular_f1_imagen(gt_boxes, gt_classes, pred_boxes, pred_classes, iou_thresh=0.5):
    """Calcula F1 para una sola imagen comparando predicciones vs GT."""
    tp = 0
    fp = 0
    fn = 0
    matched_gt = set()
    
    # Evaluar cada predicción (Buscamos si tiene match en GT)
    for i, p_box in enumerate(pred_boxes):
        best_iou = 0
        best_gt_idx = -1
        
        for j, g_box in enumerate(gt_boxes):
            # Si este GT ya fue 'encontrado', lo saltamos
            if j in matched_gt: continue
            # Si las clases no coinciden, no cuenta
            if pred_classes[i] != gt_classes[j]: continue
            
            iou = calcular_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = j
        
        # Consideramos acierto (TP) si el IoU supera el umbral
        if best_iou >= iou_thresh:
            tp += 1
            matched_gt.add(best_gt_idx)
        else:
            fp += 1 # Predicción incorrecta o duplicada
            
    # Los GT que no tuvieron match son Falsos Negativos
    fn = len(gt_boxes) - len(matched_gt)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return f1, precision, recall

### Carga de datos y visualización del Ground Truth
Utilizamos la API `pycocotools` para cargar las anotaciones oficiales de la imagen 139. Dibujamos los bounding boxes reales en color verde usando OpenCV. Esta imagen servirá como referencia absoluta para evaluar qué tan bien funcionan los modelos.

In [ ]:
# Rutas
ANN_FILE = 'datasets/coco/annotations/instances_val2017.json'
IMG_DIR = 'datasets/coco/val2017'
TARGET_ID = 139

# Inicializar API de COCO
coco = COCO(ANN_FILE)
img_info = coco.loadImgs(TARGET_ID)[0]
img_path = os.path.join(IMG_DIR, img_info['file_name'])

# Cargar imagen con OpenCV
img = cv2.imread(img_path)
img_gt = img.copy()

# Cargar anotaciones para la imagen seleccionada
ann_ids = coco.getAnnIds(imgIds=TARGET_ID)
anns = coco.loadAnns(ann_ids)

# Dibujar Ground Truth (Cajas Verdes)
for ann in anns:
    x, y, w, h = [int(v) for v in ann['bbox']]
    cat_id = ann['category_id']
    cat_name = coco.loadCats(cat_id)[0]['name']
    cv2.rectangle(img_gt, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(img_gt, f"GT: {cat_name}", (x, y-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

print(f"Imagen cargada: {img_path}")
plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB))
plt.title(f"Ground Truth (Imagen {TARGET_ID})")
plt.axis('off')
plt.show()

### Inferencia y Comparativa con F1-Score
Iteramos sobre las 5 variantes del modelo (nano, small, medium, large, x-large). Para cada uno:
* Realizamos la predicción.
* Extraemos métricas clave: número de objetos detectados, F1-Score y tiempo de inferencia.
* Generamos una tabla para comparar el rendimiento y el coste computacional de cada variante.

In [ ]:
models_dict = {
    'n': 'yolov8n.pt',
    's': 'yolov8s.pt',
    'm': 'yolov8m.pt',
    'l': 'yolov8l.pt',
    'x': 'yolov8x.pt'
}

results_data = []
best_model_name = None
max_detections = -1
best_result = None

# Extraemos cajas y nombres de clases del GT cargado en la celda anterior
gt_boxes = []
gt_names = []
for ann in anns:
    x, y, w, h = ann['bbox']
    # Convertir de [x,y,w,h] a [x1,y1,x2,y2]
    gt_boxes.append([x, y, x + w, y + h]) 
    gt_names.append(coco.loadCats(ann['category_id'])[0]['name'])

print("Ejecutando inferencia...")
for key, model_path in models_dict.items():
    print(f"Proceso modelo: YOLOv8{key}")
    model = YOLO(model_path)

    # Ejecutar inferencia
    results = model(img_path, verbose=False)
    result = results[0]

    # Preparar datos de predicción
    pred_boxes = result.boxes.xyxy.cpu().numpy()
    # Usamos nombres para comparar clases (evita lio de IDs entre COCO y YOLO)
    pred_names = [result.names[int(c)] for c in result.boxes.cls.cpu().numpy()]

    # Calcular métricas
    f1, prec, rec = calcular_f1_imagen(gt_boxes, gt_names, pred_boxes, pred_names)
    inference_speed = result.speed['inference']

    # Guardar datos
    results_data.append({
        'Modelo': f'YOLOv8{key}',
        'Detecciones': len(pred_boxes),
        'F1-Score': round(f1, 3),
        'Precision': round(prec, 3),
        'Recall': round(rec, 3),
        'Tiempo Inferencia (ms)': round(inference_speed, 2)
    })

    # Actualizar modelo con más detecciones
    if len(pred_boxes) > max_detections:
        max_detections = len(pred_boxes)
        best_model_name = key
        best_result = result

df_results = pd.DataFrame(results_data)
print("\n--- Comparativa de Rendimiento en Imagen 139 ---")
print(df_results)

### Visualización del Modelo con más detecciones
Seleccionamos automáticamente el modelo que ha detectado mayor cantidad de objetos. Dibujamos sus predicciones en rojo y generamos una figura comparativa con tres vistas:
1.  **Ground Truth (Verde):** Lo que debería detectarse.
2.  **Predicción (Rojo):** Lo que el modelo vio.
3.  **Overlay:** Superposición de ambas para verificar visualmente el ajuste de las cajas.

In [ ]:
# Preparar imagen de predicción (Rojo)
img_pred = img.copy()

# Preparar imagen Overlay (Mezcla)
img_overlay = img_gt.copy() # Ya tiene las cajas verdes

print(f"\nModelo seleccionado: YOLOv8{best_model_name}")

for box in best_result.boxes:
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    class_name = best_result.names[cls_id]

    # Dibujar en img_pred (Solo Rojo)
    cv2.rectangle(img_pred, (x1, y1), (x2, y2), (0, 0, 255), 2)
    label = f"{class_name} {conf:.2f}"
    cv2.putText(img_pred, label, (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    # Dibujar en img_overlay (Rojo sobre el Verde existente)
    cv2.rectangle(img_overlay, (x1, y1), (x2, y2), (0, 0, 255), 2)

# Mostrar resultado con 3 imagenes lado a lado
plt.figure(figsize=(24, 8))

# Ground Truth
plt.subplot(1, 3, 1)
plt.title("1. Ground Truth (Verde)")
plt.imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB))
plt.axis('off')

# Predicción
plt.subplot(1, 3, 2)
plt.title(f"2. Predicción YOLOv8{best_model_name} (Rojo)")
plt.imshow(cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB))
plt.axis('off')

# Comparación (Overlay)
plt.subplot(1, 3, 3)
plt.title("3. Superposición (GT vs Pred)")
plt.imshow(cv2.cvtColor(img_overlay, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.tight_layout()
output_dir = 'resultados'
os.makedirs(output_dir, exist_ok=True)
save_path = os.path.join(output_dir, f"comparacion_modelos_img{TARGET_ID}.png")

plt.savefig(save_path)
print(f"Imagen guardada en: {save_path}")
plt.show()